# project_tools

> MCP tools for project selection, discovery, and bookmarking

In [ ]:
#| default_exp tools.project

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

import os
from pathlib import Path
from configparser import ConfigParser

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.config import CURRENT_PROJECT, load_bookmarks, save_bookmarks
from nbdev_mcp.utils.paths import (
    expand, project_summary, is_nbdev_project, resolve_selector,
)
from nbdev_mcp.utils.paths import require_project
from nbdev_mcp.utils.rich import render_result, render_table

## Project Selection

In [ ]:
#| export
def set_project(selector: str) -> Dict[str, Any]:
    """Tool: Select an nbdev project to make it active (by path or alias).
    
    Args:
        selector: Project path, alias (prefixed with @), or bookmark name.
    
    Returns:
        Result dict with 'ok', 'project' info, and 'pretty' formatted output.
    """
    global CURRENT_PROJECT
    try:
        p = resolve_selector(selector)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    import nbdev_mcp.utils.config
    nbdev_mcp.utils.config.CURRENT_PROJECT = p
    
    meta = project_summary(p)
    pretty = render_result('Project selected', meta)
    return {'ok': True, 'project': meta, 'pretty': pretty}

In [ ]:
#| export
def current_project() -> Dict[str, Any]:
    """Tool: Show the currently active project's summary information.
    
    Returns:
        Result dict with 'ok', 'project' info, and 'pretty' formatted output.
    
    Raises:
        RuntimeError: If no project is currently selected.
    """
    p = require_project()
    meta = project_summary(p)
    pretty = render_result('Current project', meta)
    return {'ok': True, 'project': meta, 'pretty': pretty}

In [ ]:
#| export
def console_scripts_status(project: Optional[str] = None) -> Dict[str, Any]:
    """Tool: Show console_scripts entry points from settings.ini and suggest how to add them.
    
    Args:
        project: Project path or alias. Uses current project if not specified.
    
    Returns:
        Result dict with 'entries' list and suggestions for adding scripts.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    cfg = ConfigParser()
    cfg.read(p / 'settings.ini')
    cs = (cfg['DEFAULT'].get('console_scripts', '') if 'DEFAULT' in cfg else '').strip()
    entries = [e for e in cs.split() if e] if cs else []
    
    msg = (
        'No console_scripts configured. Add e.g. `console_scripts = mycli=mypkg:main` in settings.ini.'
        if not entries else 'console_scripts present.'
    )
    pretty = render_result('console_scripts', {'entries': entries or 'None'}, {})
    return {'ok': True, 'entries': entries, 'message': msg, 'pretty': pretty}

## Project Discovery

In [ ]:
#| export
def find_projects(
    roots: Optional[List[str]] = None,
    max_results: int = 50
) -> Dict[str, Any]:
    """Tool: Scan given directories (or common defaults) for nbdev projects.
    
    Searches in provided roots, NBDEV_PROJECTS env var, and common
    directories like ~/code, ~/projects, ~/repos, ~/src, ~/Dev.
    
    Args:
        roots: Additional directories to scan.
        max_results: Maximum number of projects to return (default 50).
    
    Returns:
        Result dict with 'results' list of project summaries.
    """
    search_dirs: List[Path] = []
    
    if roots:
        search_dirs += [expand(r) for r in roots]
    
    env = os.environ.get('NBDEV_PROJECTS')
    if env:
        for r in env.split(os.pathsep):
            pr = expand(r)
            if pr.exists():
                search_dirs.append(pr)
    
    home = Path.home()
    for sub in ('code', 'projects', 'repos', 'src', 'Dev', 'dev', 'Documents'):
        d = home / sub
        if d.exists():
            search_dirs.append(d)
    
    seen, results = set(), []
    for base in search_dirs:
        if not base.is_dir():
            continue
        for p in base.iterdir():
            if p.is_dir() and p not in seen:
                try:
                    if is_nbdev_project(p):
                        results.append(project_summary(p))
                        seen.add(p)
                except Exception:
                    continue
        if len(results) >= max_results:
            break
    
    rows = [[r.get('lib_name') or '?', r['project'], r['nbs_dir']] for r in results]
    pretty = render_table('Discovered nbdev projects', ['lib_name', 'path', 'nbs_dir'], rows)
    
    return {'ok': True, 'results': results, 'pretty': pretty}

## Bookmarks

In [ ]:
#| export
def bookmark_project(alias: str, path: str) -> Dict[str, Any]:
    """Tool: Bookmark an nbdev project path with a short alias name.
    
    Args:
        alias: Short name to use as bookmark (e.g., 'myproj').
        path: Path to the nbdev project directory.
    
    Returns:
        Result dict with 'alias' and 'path' of the saved bookmark.
    """
    try:
        root = resolve_selector(path)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    aliases = load_bookmarks()
    aliases[alias] = str(root)
    save_bookmarks(aliases)
    
    meta = {'alias': alias, 'path': str(root)}
    pretty = render_result('Bookmarked project', meta)
    return {'ok': True, **meta, 'pretty': pretty}

In [ ]:
#| export
def list_bookmarks() -> Dict[str, Any]:
    """Tool: List all saved project bookmarks (alias -> path).
    
    Returns:
        Result dict with 'aliases' mapping bookmark names to project paths.
    """
    aliases = load_bookmarks()
    rows = [[k, v] for k, v in aliases.items()]
    pretty = render_table('Project bookmarks', ['alias', 'path'], rows)
    return {'ok': True, 'aliases': aliases, 'pretty': pretty}

In [ ]:
#| export
def remove_bookmark(alias: str) -> Dict[str, Any]:
    """Tool: Remove a saved project bookmark by alias.
    
    Args:
        alias: Bookmark name to remove.
    
    Returns:
        Result dict with removed alias and path, or error if not found.
    """
    aliases = load_bookmarks()
    if alias in aliases:
        path = aliases.pop(alias)
        save_bookmarks(aliases)
        meta = {'alias': alias, 'removed': path}
        pretty = render_result('Removed bookmark', meta)
        return {'ok': True, **meta, 'pretty': pretty}
    return {'ok': False, 'error': f'No such alias: {alias}'}

In [ ]:
#| export
def config_status() -> Dict[str, Any]:
    """Tool: Show current nbdev-mcp configuration settings.

    Returns:
        Result dict with 'config' settings, 'bookmarks_path', and 'env_overrides'.
    """
    from nbdev_mcp.utils.config import get_config, BOOKMARKS_PATH

    cfg = get_config()
    config_data = {
        'log_level': cfg.get('log_level', 'INFO'),
        'prompt_dir': cfg.get('prompt_dir', 'prompt_templates'),
        'env_dir_name': cfg.get('env_dir_name', 'Environments'),
        'max_tree_files': cfg.get('max_tree_files', 600),
    }

    # Check for environment variable overrides
    env_overrides = {}
    for key in ['LOG_LEVEL', 'PROMPT_DIR', 'ENV_DIR_NAME', 'MAX_TREE_FILES']:
        env_key = f'NBMCP_{key}'
        if env_key in os.environ:
            env_overrides[env_key] = os.environ[env_key]

    pretty = render_result('Configuration', config_data, {
        'bookmarks_path': str(BOOKMARKS_PATH),
        'env_overrides': env_overrides or 'None'
    })

    return {
        'ok': True,
        'config': config_data,
        'bookmarks_path': str(BOOKMARKS_PATH),
        'env_overrides': env_overrides,
        'pretty': pretty
    }

In [ ]:
#| export
def prompt_templates_status() -> Dict[str, Any]:
    """Tool: List available prompt templates and their locations.

    Returns:
        Result dict with 'templates' list and 'prompt_dir' setting.
    """
    from nbdev_mcp.utils.config import get_config
    from nbdev_mcp.prompts import get_bundled_template

    cfg = get_config()
    prompt_dir = cfg.get('prompt_dir', 'prompt_templates')

    # List of expected templates
    template_names = [
        'workflow_philosophy.md',
        'nbdev_principles.md',
        'documentation_best_practices.md',
        'future_imports_guidance.md',
        'nbdev_howto.md',
        'documentation_guideline.md',
        'module_scaffold.md',
        'advanced_patterns.md',
        'main_pattern.md',
    ]

    templates = []
    for name in template_names:
        try:
            content = get_bundled_template(name)
            templates.append({
                'name': name,
                'path': f'nbdev_mcp.prompt_templates/{name}',
                'exists': True,
                'size': len(content)
            })
        except FileNotFoundError:
            templates.append({
                'name': name,
                'path': 'not found',
                'exists': False,
                'size': 0
            })

    rows = [[t['name'], t['path'], '✓' if t['exists'] else '✗'] for t in templates]
    pretty = render_table('Prompt Templates', ['name', 'path', 'exists'], rows)

    return {
        'ok': True,
        'templates': templates,
        'prompt_dir': prompt_dir,
        'pretty': pretty
    }

## MCP Registration

In [ ]:
#| export
def add_project_tools(mcp: FastMCP) -> None:
    """Attach project management tools to the MCP server.
    
    Args:
        mcp: The FastMCP server instance to register tools with.
    """
    mcp.add_tool(set_project)
    mcp.add_tool(current_project)
    mcp.add_tool(console_scripts_status)
    mcp.add_tool(find_projects)
    mcp.add_tool(bookmark_project)
    mcp.add_tool(list_bookmarks)
    mcp.add_tool(remove_bookmark)
    mcp.add_tool(config_status)
    mcp.add_tool(prompt_templates_status)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()